# 33 — Screener tín hiệu kỹ thuật toàn sàn

**Sản phẩm 3.** Quét toàn bộ cổ phiếu HOSE tìm sáu tín hiệu kỹ thuật, chấm điểm
và xếp hạng — chạy sau mỗi phiên, mất khoảng một phút.

Ba nguyên tắc quyết định code trông thế nào:

| Nguyên tắc | Vì sao |
|---|---|
| Mọi phép `shift`/`diff` phải **trong từng mã** | quên `groupby` là để giá mã sau rò vào mã trước |
| Tín hiệu chỉ dùng dữ liệu **tới hết phiên đó** | nếu không, kết quả backtest là ảo |
| Mỗi tín hiệu phải kèm **tần suất nền** | tín hiệu kêu 40% số phiên thì không phải tín hiệu |

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, hom_nay, lui_ngay, nen
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Dữ liệu và vũ trụ

In [2]:
danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")

# Cần đủ lịch sử cho MA200 cộng vùng warm-up — lấy 2 năm cho chắc
gia = client.eod.stock.ohlcv(danh_muc["symbol"].tolist(), start=lui_ngay(HOM_NAY, nam=2))
gia = gia.sort_values(["symbol", "date"]).reset_index(drop=True)

PHIEN = gia["date"].max()
print(f"{gia['symbol'].nunique()} mã · {len(gia):,} dòng · phiên gần nhất {PHIEN:%d/%m/%Y}")

405 mã · 198,524 dòng · phiên gần nhất 11/08/2026


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


In [3]:
# Lọc thanh khoản: giá (nghìn VND) × khối lượng × 1.000 = VND
gia = gia.assign(gtgd=gia["close"] * gia["volume"] * 1_000)
gtgd_bq = (
    gia[gia["date"] > PHIEN - pd.Timedelta(days=60)]
    .groupby("symbol", observed=True)["gtgd"]
    .mean()
)
VU_TRU = gtgd_bq[gtgd_bq >= 5e9].index

bang = gia[gia["symbol"].isin(VU_TRU)].copy()
print(f"Sau lọc GTGD ≥ 5 tỷ/phiên: {bang['symbol'].nunique()} mã, {len(bang):,} dòng")

Sau lọc GTGD ≥ 5 tỷ/phiên: 141 mã, 68,568 dòng


## 2 · Tính chỉ báo — một lời gọi cho cả bảng

Đây là chỗ tầng `df.finlens.*` trả công: một chuỗi method chạy trên cả bảng
140 mã và **tự tách nhóm**. Không vòng lặp, không `groupby().apply()`.

In [4]:
import time

t0 = time.perf_counter()
bang = (
    bang.finlens.sma(20)
    .finlens.sma(50)
    .finlens.sma(200)
    .finlens.rsi(14)
    .finlens.macd()
    .finlens.atr(14)
)
print(f"6 chỉ báo trên {len(bang):,} dòng · {bang['symbol'].nunique()} mã: {time.perf_counter() - t0:.1f} giây")
print([c for c in bang.columns if c.startswith(("sma", "rsi", "macd", "atr"))])

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\IPython\core\interactiveshell.py:3715: DataQualityWarning: `SMA` cần ít nhất 200 dòng mỗi nhóm nhưng 3 nhóm (`symbol`) ngắn hơn thế: GEL, VCK, VPX. Chúng ra TOÀN `NaN` — TA-Lib không báo lỗi cho trường hợp này.
  if await self.run_code(code, result, async_=asy):


6 chỉ báo trên 68,568 dòng · 141 mã: 0.1 giây
['sma_20', 'sma_50', 'sma_200', 'rsi_14', 'macd_12_26_9', 'macdsignal_12_26_9', 'macdhist_12_26_9', 'atr_14']


## 3 · Các phép biến đổi phụ — luôn `groupby`

Đây là chỗ dễ sai nhất khi tự viết. `shift`, `diff`, `rolling` trên một bảng
dạng long **phải** đi qua `groupby("symbol")`. Quên nó là tái tạo đúng cái lỗi
rò rỉ ranh giới nhóm mà notebook `31` đã đo.

In [5]:
g = bang.groupby("symbol", observed=True)

bang["close_truoc"] = g["close"].shift(1)
bang["rsi_truoc"] = g["rsi_14"].shift(1)
bang["macdhist_truoc"] = g["macdhist_12_26_9"].shift(1)
bang["sma20_truoc"] = g["sma_20"].shift(1)
bang["sma50_truoc"] = g["sma_50"].shift(1)

# Đỉnh 20 phiên TRƯỚC phiên hiện tại — closed="left" để phiên hôm nay không tự
# tính vào đỉnh của chính nó. Thiếu chi tiết này thì "vượt đỉnh" không bao giờ đúng.
bang["dinh_20"] = g["close"].transform(
    lambda s: s.rolling(20, min_periods=20, closed="left").max()
)
bang["kl_bq_20"] = g["volume"].transform(
    lambda s: s.rolling(20, min_periods=20, closed="left").mean()
)

print("Kiểm tra ranh giới nhóm — dòng đầu của mỗi mã phải có close_truoc là NaN:")
dau_moi_ma = bang.groupby("symbol", observed=True).head(1)
print(f"  {dau_moi_ma['close_truoc'].isna().sum()} / {len(dau_moi_ma)} mã đúng như vậy")

Kiểm tra ranh giới nhóm — dòng đầu của mỗi mã phải có close_truoc là NaN:
  141 / 141 mã đúng như vậy


## 4 · Sáu tín hiệu

Mỗi tín hiệu là một cột boolean. Định nghĩa viết ra rõ ràng để có thể tranh
luận được — một screener mà không ai đọc được điều kiện là một hộp đen.

In [6]:
TIN_HIEU = {
    "RSI thoát quá bán": (bang["rsi_truoc"] < 30) & (bang["rsi_14"] >= 30),
    "MACD cắt lên": (bang["macdhist_truoc"] < 0) & (bang["macdhist_12_26_9"] >= 0),
    "Cắt vàng MA20/50": (bang["sma20_truoc"] <= bang["sma50_truoc"]) & (bang["sma_20"] > bang["sma_50"]),
    "Vượt đỉnh 20 phiên": bang["close"] > bang["dinh_20"],
    "Khối lượng đột biến": bang["volume"] > 2 * bang["kl_bq_20"],
    "Trên MA200": bang["close"] > bang["sma_200"],
}

for ten, dieu_kien in TIN_HIEU.items():
    bang[ten] = dieu_kien.fillna(False)

### Tần suất nền — bước không được bỏ

Trước khi tin bất kỳ tín hiệu nào, hỏi: nó kêu bao nhiêu lần?

In [7]:
tan_suat = pd.DataFrame(
    {
        "tần suất %": [bang[t].mean() * 100 for t in TIN_HIEU],
        "số lần": [int(bang[t].sum()) for t in TIN_HIEU],
    },
    index=list(TIN_HIEU),
).round(2)
tan_suat.sort_values("tần suất %", ascending=False)

,tần suất %,số lần
Trên MA200,32.63,22373
Vượt đỉnh 20 phiên,9.87,6770
Khối lượng đột biến,7.88,5404
MACD cắt lên,3.58,2458
RSI thoát quá bán,1.29,885
Cắt vàng MA20/50,0.94,645


Bảng này chia sáu điều kiện thành hai loại khác hẳn nhau.

`Trên MA200` kêu ở khoảng một phần ba số phiên — nó mô tả **trạng thái** của mã
chứ không đánh dấu một sự kiện. Các điều kiện giao cắt (`RSI thoát quá bán`,
`Cắt vàng MA20/50`) kêu quanh 1% số phiên; đó mới là sự kiện.

Phân biệt hai loại này quan trọng, vì cộng chúng vào cùng một điểm số là để một
bộ lọc trạng thái — thứ đúng với một phần ba thị trường ở mọi thời điểm — lấn
át những sự kiện hiếm mà bạn thật sự đang tìm.

## 5 · Bảng kết quả cho phiên gần nhất

In [8]:
SU_KIEN = ["RSI thoát quá bán", "MACD cắt lên", "Cắt vàng MA20/50", "Vượt đỉnh 20 phiên", "Khối lượng đột biến"]
BO_LOC = ["Trên MA200"]

hom_nay_bang = bang[bang["date"] == PHIEN].copy()
hom_nay_bang["số tín hiệu"] = hom_nay_bang[SU_KIEN].sum(axis=1)

ket_qua = (
    hom_nay_bang[hom_nay_bang["số tín hiệu"] > 0]
    .merge(danh_muc[["symbol", "short_name", "icb_name2"]], on="symbol")
    .merge(gtgd_bq.div(1e9).rename("gtgd_ty").reset_index(), on="symbol")
    .sort_values(["số tín hiệu", "gtgd_ty"], ascending=[False, False])
)

print(f"Phiên {PHIEN:%d/%m/%Y}: {len(ket_qua)}/{hom_nay_bang['symbol'].nunique()} mã có ít nhất một tín hiệu")
print(ket_qua["số tín hiệu"].value_counts().sort_index(ascending=False).rename("số mã").to_frame().to_string())

Phiên 11/08/2026: 17/141 mã có ít nhất một tín hiệu
             số mã
số tín hiệu       
3                1
2                2
1               14


In [9]:
COT = ["symbol", "short_name", "icb_name2", "số tín hiệu", *SU_KIEN, *BO_LOC, "close", "rsi_14", "gtgd_ty"]

hien = ket_qua.head(25)[COT].copy()
for c in [*SU_KIEN, *BO_LOC]:
    hien[c] = hien[c].map({True: "✓", False: ""})
hien.assign(rsi_14=lambda d: d["rsi_14"].round(1), gtgd_ty=lambda d: d["gtgd_ty"].round(1)).reset_index(drop=True)

,symbol,short_name,icb_name2,số tín hiệu,RSI thoát quá bán,MACD cắt lên,Cắt vàng MA20/50,Vượt đỉnh 20 phiên,Khối lượng đột biến,Trên MA200,close,rsi_14,gtgd_ty
0,FRT,Bán lẻ FPT,Bán lẻ,3,,,✓,✓,✓,✓,146.50,79.0,64.2
1,HCM,Chứng khoán HSC,Dịch vụ tài chính,2,,✓,,✓,,✓,26.85,66.5,246.3
2,HHP,HHP Global,Tài nguyên Cơ bản,2,,,,✓,✓,✓,16.45,74.7,14.9
3,FPT,FPT Corp,Công nghệ Thông tin,1,,,,✓,,,71.90,60.7,533.9
4,MBB,MBBank,Ngân hàng,1,✓,,,,,,20.45,32.1,331.9
5,VNM,VINAMILK,Thực phẩm và đồ uống,1,,,,✓,,✓,62.20,65.7,250.1
6,ORS,Chứng khoán Tiên Phong,Dịch vụ tài chính,1,,,,✓,,✓,14.50,59.7,78.6
7,PVT,Vận tải Dầu khí PVTrans,Hàng & Dịch vụ Công nghiệp,1,,,,✓,,✓,19.70,61.3,57.6
8,OCB,Ngân hàng Phương Đông,Ngân hàng,1,,,,,✓,✓,10.80,57.7,40.7
9,DGW,Thế Giới Số,Bán lẻ,1,,,,✓,,,41.40,65.4,32.4


⚠️ Cột `Trên MA200` để riêng, **không** cộng vào `số tín hiệu`. Nó trả lời câu
hỏi khác: "tín hiệu này xuất hiện trong xu hướng tăng hay trong một cú hồi giữa
xu hướng giảm?" Đó là bối cảnh, không phải sự kiện.

## 6 · Tín hiệu nào đang phổ biến — và ở ngành nào

In [10]:
theo_tin_hieu = pd.DataFrame(
    {"số mã": [int(hom_nay_bang[t].sum()) for t in SU_KIEN]}, index=SU_KIEN
).reset_index()
theo_tin_hieu.columns = ["tín hiệu", "số mã"]

bar_ngang(
    theo_tin_hieu,
    nhan="tín hiệu",
    gia_tri="số mã",
    tieu_de=f"Số mã phát tín hiệu — phiên {PHIEN:%d/%m/%Y}",
    phu_de=f"Trên {hom_nay_bang['symbol'].nunique()} mã HOSE đã qua lọc thanh khoản",
    nhan_x="số mã",
    dinh_dang_nhan="{:.0f}",
)

In [11]:
theo_nganh = (
    ket_qua.groupby("icb_name2", observed=True)
    .agg(so_ma_co_tin_hieu=("symbol", "count"), tong_tin_hieu=("số tín hiệu", "sum"))
    .sort_values("tong_tin_hieu", ascending=False)
    .reset_index()
)

bar_ngang(
    theo_nganh,
    nhan="icb_name2",
    gia_tri="tong_tin_hieu",
    tieu_de="Tổng số tín hiệu theo ngành",
    phu_de="Nhiều tín hiệu tập trung ở một ngành thường là cả ngành đang chuyển động cùng nhau",
    nhan_x="tổng tín hiệu",
    dinh_dang_nhan="{:.0f}",
)

## 7 · Tín hiệu này có nghĩa gì không? — đo trên lịch sử

Cùng phương pháp event study của notebook `32`, lần này cho tín hiệu chỉ báo.
**Đây là phần phân biệt một screener với một máy sinh số ngẫu nhiên.**

In [12]:
KHUNG = [1, 5, 10, 20]

for k in KHUNG:
    bang[f"ls_{k}"] = (
        bang.groupby("symbol", observed=True)["close"].shift(-k) / bang["close"] - 1
    ) * 100

nen_tb = {k: bang[f"ls_{k}"].mean() for k in KHUNG}
print("Đường nền — lợi suất trung bình của mọi phiên:")
for k, v in nen_tb.items():
    print(f"  {k:>2} phiên: {v:+.3f}%")

Đường nền — lợi suất trung bình của mọi phiên:
   1 phiên: +0.054%
   5 phiên: +0.268%
  10 phiên: +0.475%
  20 phiên: +0.954%


In [13]:
danh_gia = []
for ten in SU_KIEN:
    co = bang[bang[ten]]
    dong = {"tín hiệu": ten, "số lần": len(co)}
    for k in KHUNG:
        r = co[f"ls_{k}"].dropna()
        if len(r) < 100:
            dong[f"{k}p"] = np.nan
            dong[f"{k}p / ss"] = np.nan
            continue
        vuot = r.mean() - nen_tb[k]
        ss = r.std() / np.sqrt(len(r))
        dong[f"{k}p"] = round(vuot, 3)
        dong[f"{k}p / ss"] = round(vuot / ss, 2)
    danh_gia.append(dong)

hieu_qua = pd.DataFrame(danh_gia).sort_values("20p", ascending=False)
print("Lợi suất trung bình vượt nền (điểm phần trăm) và tỷ số trên sai số chuẩn.")
print("|tỷ số| < 2 = không phân biệt được với nhiễu.\n")
hieu_qua

Lợi suất trung bình vượt nền (điểm phần trăm) và tỷ số trên sai số chuẩn.
|tỷ số| < 2 = không phân biệt được với nhiễu.



,tín hiệu,số lần,1p,1p / ss,5p,5p / ss,10p,10p / ss,20p,20p / ss
3,Vượt đỉnh 20 phiên,6770,0.202,6.12,0.442,5.53,1.133,9.78,1.881,10.62
0,RSI thoát quá bán,885,0.491,5.30,0.578,3.14,0.656,2.53,1.629,4.56
4,Khối lượng đột biến,5404,0.199,5.09,0.222,2.52,0.530,4.18,1.295,6.77
2,Cắt vàng MA20/50,645,0.056,0.72,-0.071,-0.39,0.386,1.37,-0.047,-0.11
1,MACD cắt lên,2458,0.147,3.30,-0.061,-0.58,-0.121,-0.75,-0.164,-0.67


In [14]:
ve = hieu_qua.melt(
    id_vars="tín hiệu",
    value_vars=[f"{k}p" for k in KHUNG],
    var_name="khung",
    value_name="vuot_nen",
).dropna()

fig = go.Figure()
for i, k in enumerate(KHUNG):
    con = ve[ve["khung"] == f"{k}p"]
    fig.add_trace(
        go.Bar(
            x=con["tín hiệu"],
            y=con["vuot_nen"],
            name=f"{k} phiên",
            marker=dict(color=CHUOI[i], line=dict(color="#fcfcfb", width=2)),
            hovertemplate="%{x}<br>%{y:+.3f} đpt<extra>" + f"{k} phiên</extra>",
        )
    )
fig.add_hline(y=0, line_width=1, line_color="#c3c2b7")
fig.update_layout(
    barmode="group",
    title_text="Lợi suất vượt nền sau mỗi tín hiệu<br>"
    "<sub style='color:#52514e'>Hai năm dữ liệu HOSE · chưa trừ chi phí giao dịch</sub>",
    yaxis_title="điểm phần trăm so với nền",
    height=520,
    xaxis=dict(tickangle=-20),
)
fig

### Đọc kết quả này thế nào

Nhìn cột `/ ss` trước, rồi mới nhìn cột giá trị. Một tín hiệu chỉ đáng chú ý
khi **cả hai** đều thuận: chênh lệch đủ lớn để bù chi phí giao dịch (khoảng
0,3–0,4% cho một vòng mua-bán ở Việt Nam), *và* tỷ số trên sai số chuẩn vượt 2.

Kết quả tách rõ thành hai nhóm. Ba điều kiện thiên về **đà tăng** —
`Vượt đỉnh 20 phiên`, `Khối lượng đột biến`, `RSI thoát quá bán` — cho chênh
lệch dương ở khung 20 phiên với tỷ số vượt 4, tức là vượt cả ngưỡng chi phí lẫn
ngưỡng nhiễu. Hai điều kiện **giao cắt đường trung bình** — `Cắt vàng MA20/50`
và `MACD cắt lên` — không phân biệt được với nhiễu ở mọi khung.

Điều đó hợp lý: giao cắt MA là một hàm của giá quá khứ đã được làm trơn, nên
tới lúc nó cắt thì phần lớn chuyển động đã xảy ra rồi.

⚠️ Ba giới hạn phải nhớ trước khi mang con số này đi đâu:

1. **Hai năm là một chế độ thị trường**, không phải nhiều chế độ. Một tín hiệu
   đà tăng gần như luôn đẹp trong giai đoạn thị trường lên; câu hỏi thật là nó
   sống thế nào qua một đợt giảm sâu.
2. **Chưa trừ chi phí, và chưa tính trượt giá.** Bảng trên là lợi suất gộp.
3. **Vũ trụ được lọc bằng thanh khoản của *hôm nay***, tức là có thiên lệch
   sống sót: các mã mất thanh khoản trong hai năm qua đã bị loại khỏi mẫu.
   Notebook `34` dựng bộ khung tính lại bộ lọc theo từng phiên quá khứ.

## 8 · Nhìn tận mắt mã dẫn đầu

In [15]:
if len(ket_qua):
    MA_XEM = ket_qua.iloc[0]["symbol"]
    tin_hieu_co = [t for t in SU_KIEN if ket_qua.iloc[0][t]]
    print(f"{MA_XEM} — {ket_qua.iloc[0]['short_name']}")
    print(f"Tín hiệu: {', '.join(tin_hieu_co)}")
    print(f"Trên MA200: {'có' if ket_qua.iloc[0]['Trên MA200'] else 'không'}")

    xem = bang[(bang["symbol"] == MA_XEM) & (bang["date"] > PHIEN - pd.Timedelta(days=180))]

    fig = nen(xem, tieu_de=f"{MA_XEM} — 6 tháng", phu_de=" · ".join(tin_hieu_co))
    for cot_ma, mau, nhan in [("sma_20", CHUOI[0], "SMA 20"), ("sma_50", CHUOI[1], "SMA 50")]:
        fig.add_trace(
            go.Scatter(x=xem["date"], y=xem[cot_ma], name=nhan, line=dict(width=2, color=mau)),
            row=1,
            col=1,
        )
    fig.update_layout(showlegend=True)
    fig.show()

FRT — Bán lẻ FPT
Tín hiệu: Cắt vàng MA20/50, Vượt đỉnh 20 phiên, Khối lượng đột biến
Trên MA200: có


## 9 · Xuất kết quả

In [16]:
THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)

xuat = ket_qua[COT].copy()
for c in [*SU_KIEN, *BO_LOC]:
    xuat[c] = xuat[c].astype(int)

tep = THU_MUC_RA / f"screener_tin_hieu_{PHIEN:%Y%m%d}.xlsx"
xuat.round(2).to_excel(tep, index=False, sheet_name="Tin hieu")
print(f"Đã ghi {len(xuat)} dòng → {tep.name}")

Đã ghi 17 dòng → screener_tin_hieu_20260811.xlsx


## Tổng kết

| Bước | Chi tiết dễ bỏ sót |
|---|---|
| Tính chỉ báo | `df.finlens.*` tự tách nhóm — đừng gọi TA-Lib thẳng |
| `shift` / `diff` / `rolling` | **phải** qua `groupby("symbol")` |
| Đỉnh N phiên | `closed="left"`, nếu không phiên hôm nay tự tính vào đỉnh của nó |
| Lọc thanh khoản | trước mọi xếp hạng |
| Tần suất nền | tín hiệu kêu 60% số phiên là bộ lọc trạng thái, không phải sự kiện |
| Đánh giá lịch sử | chênh lệch **và** tỷ số trên sai số chuẩn, cả hai |

**Và điều cuối:** danh sách ở trên là *đầu vào cho việc phân tích*, không phải
đầu ra của nó. Một mã có bốn tín hiệu vẫn có thể đang trong một câu chuyện mà
biểu đồ không kể — kết quả kinh doanh sắp công bố, một đợt phát hành, một tin
ngành. Screener thu hẹp; con người quyết định.

---

**Tiếp theo:** [`34_backtest_chien_luoc.ipynb`](34_backtest_chien_luoc.ipynb) —
backtest có kỷ luật: tránh look-ahead, tính chi phí, và so với chuẩn.